In [ ]:
import mlbstatsapi
import requests
import sqlite3
import time
import copy
import threading
from concurrent.futures import ThreadPoolExecutor

from datetime import datetime
from dataclasses import dataclass
from pybaseball import  playerid_lookup

In [ ]:
mlb = mlbstatsapi.Mlb()

In [ ]:
#GET GAMES IN DATE RANGE

"https://statsapi.mlb.com/api/v1/schedule/games/?sportId=1&startDate=2025-06-21&endDate=2025-06-22"

In [ ]:
players = mlb.get_people(1)

In [ ]:
print(players[0].id)

In [ ]:
mlb.get_people_id("Cal Raleigh")

In [ ]:
teamIdToAbbreviation = {
    136: "SEA",
    111: "BOS",
    158: "MIL",
    141: "TOR",
    112: "CHN",
    113: "CIN",
    115: "COL",
    140: "TEX",
    109: "ARI",
    144: "ATL",
    117: "HOU",
    142: "MIN",
    133: "OAK",
    118: "KCA",
    110: "BAL",
    147: "NYA",
    120: "WAS",
    108: "ANA",
    145: "CHA",
    146: "MIA",
    139: "TBA",
    143: "PHI",
    116: "DET",
    121: "NYN",
    119: "LAN",
    134: "PIT",
    137: "SFN",
    138: "SLN",
    135: "SDN",
    114: "CLE",
}

In [ ]:
venueIdToAbbreviation = {
    5150: "SEO01",
    15  : "PHO01",
    2   : "BAL12",
    4   : "CHI12",
    2602: "CIN09",
    2392: "HOU03",
    7   : "KAN06",
    22  : "LOS03",
    4169: "MIA02",
    10  : "OAK01",
    2680: "SAN02",
    680 : "SEA03",
    12  : "STP01",
    5325: "ARL03",
    3289: "NYC20",
    2681: "PHI13",
    17  : "CHI11",
    3309: "WAS11",
    32  : "MIL06",
    3312: "MIN04",
    2889: "STL10",
    1   : "ANA01",
    4705: "ATL03",
    19  : "DEN02",
    2394: "DET05",
    3313: "NYC21",
    31  : "PIT08",
    2395: "SFO03",
    5   : "CLE08",
    14  : "TOR02",
    3   : "BOS07",
    5340: "MEX02",
    5381: "LON01",
    3949: "BIR01",
    5027: "WIL02",
    2529: "SAC01",
    2397: "TOK01",
    2523: "TAM02"
}

In [ ]:
def findJsonField(curNode, curNodeName, curTraversal, hiddenNodeName, traversals):
    if curNodeName == hiddenNodeName:
        traversals.append(curTraversal)

    if isinstance(curNode, dict):
        for key in curNode:
            nestedTraversal = copy.deepcopy(curTraversal)
            
            nestedTraversal.append(key)
            
            findJsonField(curNode[key], key, nestedTraversal, hiddenNodeName, traversals)

#traversals = []
#
#findJsonField(gameJson, "None", [], "pitchers", traversals)
#
#for traversal in traversals:
#    print(traversal)

In [ ]:
databaseLock = threading.Lock()

In [ ]:
def getRetroIdFromFirstAndLastName(firstName, lastName):
    firstName = firstName.lower()
    lastName  = lastName .lower()
    
    retroId = ""
    
    if len(lastName) < 4:
        charactersToFill = 4 - len(lastName)
        retroId          = lastName + ("-" * charactersToFill)
    else:
        retroId = lastName[:4]

    retroId = retroId + firstName[:1] + "0"

    similarPlayers = 0
    
    findExistingPlayerQuery = f"SELECT * FROM bios WHERE PLAYERID LIKE \"%{retroId}%\""

    with databaseLock:
        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()
    
        cursor.execute(findExistingPlayerQuery)
    
        rows = cursor.fetchall()
    
        similarPlayers = len(rows)
    
        cursor    .close()
        connection.close()

    if similarPlayers + 1 > 9:
        retroId = retroId + str(similarPlayers + 1)
    else:
        retroId = retroId + "0" + str(similarPlayers + 1)

    return retroId

In [ ]:
def insertRetroIdIntoBiosAndChadwick(mlbUuid, retroId, firstName, lastName):
    biosQuery  = "INSERT INTO bios (PLAYERID, LAST, FIRST, NICKNAME, BIRTHDATE, BIRTH_CITY, BIRTH_STATE, "
    biosQuery += "BIRTH_COUNTRY, PLAY_DEBUT, PLAY_LASTGAME, MGR_DEBUT, MGR_LASTGAME, COACH_DEBUT, COACH_LASTGAME, "
    biosQuery += "UMP_DEBUT, UMP_LASTGAME, DEATHDATE, DEATH_CITY, DEATH_STATE, DEATH_COUNTRY, BATS, THROWS, HEIGHT, "
    biosQuery += "CEMETERY, CEME_CITY, CEME_STATE, CEME_COUNTRY, CEME_NOTE, BIRTH_NAME, NAME_CHG, BAT_CHG, HOF, MANUALLY_GENERATED) "
    biosQuery += f"VALUES(\"{retroId}\", \"{lastName}\", \"{firstName}\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", "
    biosQuery += "\"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", "
    biosQuery += "\"\", \"\", \"\", \"\", 1)"
    
    chadwickQuery = f"UPDATE chadwick SET key_retro = \"{retroId}\", key_retro_manually_generated = 1 WHERE key_uuid = \"{mlbUuid}\""

    with databaseLock:
        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()
    
        cursor.execute(biosQuery    )
        cursor.execute(chadwickQuery)
        
        connection.commit()
    
        cursor    .close()
        connection.close()

In [ ]:
def getRetroIdFromMlbId(mlbId):
    if mlbId == -1:
        return -1
    
    retroQuery = f"SELECT key_retro,name_first,name_last,key_uuid FROM chadwick WHERE key_mlbam = \"{mlbId}\""

    retroId   = ""
    firstName = ""
    lastName  = ""
    
    with databaseLock:
        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()
    
        cursor.execute(retroQuery)
    
        rows = cursor.fetchall()
        
        if len(rows) > 1:
            return ""
        elif len(rows) < 1:
            return ""
        else:        
            playerRow = rows     [0]
            retroId   = playerRow[0]
    
            playerRow = rows     [0]
            firstName = playerRow[1]
            lastName  = playerRow[2]
    
        cursor    .close()
        connection.close()

    if retroId == "":
        print(mlbId)
        
        retroId = getRetroIdFromFirstAndLastName(firstName, lastName)

        key_uuid = playerRow[3]

        insertRetroIdIntoBiosAndChadwick(key_uuid, retroId, firstName, lastName)

    return retroId

In [ ]:
def getGameType(gameJson):
    gameData = gameJson["gameData"]
    game     = gameData["game"    ]
    gameType = game    ["type"    ]

    match gameType:
        case "R":
            return "regular"
        case "A":
            return "allstar"
        case "W":
            return "wildcard"
        case "D":
            return "divisionseries"
        case "L":
            return "lcs"
        case "W":
            return "worldseries"
        case _:
            return "unknown"
    
    return gameType

In [ ]:
def getUsedDh(batters):
    for batter in batters:
        if batter.designatedHitter == "1":
            return "true"

    return "false"

In [ ]:
#TODO: I think retrosheet is kinda wrong on this one...
def getTieBreaker(gameJson):
    #gameData   = gameJson["gameData"  ]
    #game       = gameData["game"      ]
    #tieBreaker = game    ["tiebreaker"]
#
    #if tieBreaker == "N":
    #    return ""
    #else:
    return "2"

In [ ]:
def getUmpires(gameJson):
    officialTypeToId = {}
    
    liveData  = gameJson["liveData" ]
    boxScore  = liveData["boxscore" ]
    officials = boxScore["officials"]

    for official in officials:
        mlbId        = official["official"]["id"]
        officialType = official["officialType"]

        retroId = getRetroIdFromMlbId(mlbId)

        officialTypeToId[officialType] = retroId

    return officialTypeToId

In [ ]:
def getWinningLosingAndSavingPitcher(gameJson):
    liveData  = gameJson["liveData" ]
    decisions = liveData["decisions"]

    #TODO: What to do when any of these are empty
    winningPitcher = decisions["winner"]
    losingPitcher  = decisions["loser" ]
    savingPitcher  = None

    if "save" in decisions:
        savingPitcher = decisions["save"]

    winningPitcherId = winningPitcher["id"]
    losingPitcherId  = losingPitcher ["id"]

    savingPitcherId = ""

    if savingPitcher is not None:
        savingPitcherId = savingPitcher["id"]

    winningPitcherRetroId = getRetroIdFromMlbId(winningPitcherId)
    losingPitcherRetroId  = getRetroIdFromMlbId(losingPitcherId )
    savingPitcherRetroId  = getRetroIdFromMlbId(savingPitcherId )

    return winningPitcherRetroId, losingPitcherRetroId, savingPitcherRetroId

In [ ]:
def getTeamAbbreviation(gameJson, isHome):
    if isHome:
        homeTeamId = gameJson["liveData"]["boxscore"]["teams"]["home"]["team"]["id"]

        return teamIdToAbbreviation[homeTeamId]
    else:
        awayTeamId = gameJson["liveData"]["boxscore"]["teams"]["away"]["team"]["id"]

        return teamIdToAbbreviation[awayTeamId]

In [ ]:
def getGameNumber(gameJson):
    gameData     = gameJson["gameData"    ]
    game         = gameData["game"        ]
    doubleHeader = game    ["doubleHeader"]
    gameNumber   = game    ["gameNumber"  ]
    
    modifiedGameNumber = 0

    if doubleHeader == "S":
        modifiedGameNumber = gameNumber

    return modifiedGameNumber

In [ ]:
def getGameId(gameJson, homeTeamAbbreviation, gameNumber):
    gameData     = gameJson["gameData"    ]
    dateTime     = gameData["datetime"    ]
    officialDate = dateTime["officialDate"]

    strippedDate = officialDate.replace("-", "")

    gameId = homeTeamAbbreviation + strippedDate + str(gameNumber)

    return gameId

In [ ]:
def getGameAttendance(gameJson):
    gameData   = gameJson["gameData"  ]
    gameInfo   = gameData["gameInfo"  ]
    attendance = gameInfo["attendance"]

    return attendance

In [ ]:
def getGameDuration(gameJson):
    gameData     = gameJson["gameData"           ]
    gameInfo     = gameData["gameInfo"           ]
    gameDuration = gameInfo["gameDurationMinutes"]

    return gameDuration

In [ ]:
def getOfficialScorer(gameJson):
    gameData      = gameJson ["gameData"      ]
    offScorer     = gameData ["officialScorer"]
    offScorerId   = offScorer["id"            ]

    retroId = getRetroIdFromMlbId(offScorerId)

    if retroId == -1:
        firstNameLastName = offScorer["fullName"].split()
        
        firstName = firstNameLastName[0]
        lastName  = firstNameLastName[1]
        
        retroId = getRetroIdFromFirstAndLastName(firstName, lastName)

    return retroId

In [ ]:
def getInningsPlayed(gameJson):
    liveData  = gameJson["liveData" ]
    lineScore = liveData["linescore"]

    inningsPlayed = lineScore["currentInning"]

    return inningsPlayed

In [ ]:
def getWinningTeam(gameJson):
    liveData = gameJson["liveData"]
    boxScore = liveData["boxscore"]

    teams    = boxScore["teams"]
    awayTeam = teams["away"]
    homeTeam = teams["home"]

    awayTeamStats = awayTeam["teamStats"]
    homeTeamStats = homeTeam["teamStats"]

    awayTeamAbbr = teamIdToAbbreviation[awayTeam["team"]["id"]]
    homeTeamAbbr = teamIdToAbbreviation[homeTeam["team"]["id"]]

    awayTeamBattingStats = awayTeamStats["batting"]
    homeTeamBattingStats = homeTeamStats["batting"]

    awayTeamRuns = awayTeamBattingStats["runs"]
    homeTeamRuns = homeTeamBattingStats["runs"]

    if awayTeamRuns > homeTeamRuns:
        return awayTeamAbbr
    elif homeTeamRuns > awayTeamRuns:
        return homeTeamAbbr
    else:
        return "TIE"

In [ ]:
def getLosingTeam(gameJson):
    liveData = gameJson["liveData"]
    boxScore = liveData["boxscore"]

    teams    = boxScore["teams"]
    awayTeam = teams["away"]
    homeTeam = teams["home"]

    awayTeamStats = awayTeam["teamStats"]
    homeTeamStats = homeTeam["teamStats"]

    awayTeamAbbr = teamIdToAbbreviation[awayTeam["team"]["id"]]
    homeTeamAbbr = teamIdToAbbreviation[homeTeam["team"]["id"]]

    awayTeamBattingStats = awayTeamStats["batting"]
    homeTeamBattingStats = homeTeamStats["batting"]

    awayTeamRuns = awayTeamBattingStats["runs"]
    homeTeamRuns = homeTeamBattingStats["runs"]

    if awayTeamRuns > homeTeamRuns:
        return homeTeamAbbr
    elif homeTeamRuns > awayTeamRuns:
        return awayTeamAbbr
    else:
        return "TIE"

In [ ]:
def getVenue(gameJson):
    gameData = gameJson["gameData"]
    venue    = gameData["venue"   ]

    venueId           = venue["id"]
    venueAbbreviation = ""

    if venueId in venueIdToAbbreviation:
        venueAbbreviation = venueIdToAbbreviation[venueId]
    else:
        venueAbbreviation = "UKN"

    return venueAbbreviation

In [ ]:
def getGameTime(gameJson):
    gameData = gameJson["gameData"]
    dateTime = gameData["datetime"]
    time     = dateTime["time"]
    amPm     = dateTime["ampm"]

    gameTime = time + amPm

    return gameTime

In [ ]:
def getDayOrNight(gameJson):
    gameData = gameJson["gameData"]
    dateTime = gameData["datetime"]
    dayNight = dateTime["dayNight"]

    return dayNight

In [ ]:
mlbConditionToRetroCondition = {
    "Roof Closed"        : "dome",
    "Dome"               : "dome",
    "Partly Cloudy"      : "overcast",
    "Clear"              : "sunny",
    "Cloudy"             : "overcast",
    "Controlled Climate" : "dome",
    "Haze"               : "overcast",
    "Light Rain"         : "overcast",
    "Drizzle"            : "overcast",
    "Mist"               : "overcast",
    "Mostly Cloudy"      : "cloudy",
    "Mostly Sunny"       : "sunny",
    "Overcast"           : "overcast",
    "Partly Cloudy"      : "overcast",
    "Rain"               : "cloudy",
    "Sunny"              : "sunny"
}

def getGameCondition(gameJson):
    gameData  = gameJson["gameData" ]
    weather   = gameData["weather"  ]
    condition = weather ["condition"]

    if condition in mlbConditionToRetroCondition:
        return mlbConditionToRetroCondition[condition]
    else:
        return "unknown"

In [ ]:
mlbPrecipitationToRetroPrecipitation = {
    "Light Rain": "rain",
    "Rain"      : "rain",
    "Drizzle"   : "drizzle"
}

def getGamePrecipitation(gameJson):
    gameData  = gameJson["gameData" ]
    weather   = gameData["weather"  ]
    condition = weather ["condition"]

    if condition in mlbPrecipitationToRetroPrecipitation:
        return mlbPrecipitationToRetroPrecipitation[condition]
    else:
        return "none"

In [ ]:
def getGameTemp(gameJson):
    gameData  = gameJson["gameData" ]
    weather   = gameData["weather"  ]
    temp      = weather ["temp"     ]

    return temp

In [ ]:
mlbWindDirectionToRetroWindDirection = {
    "In From CF"  : "fromcf",
    "In From LF"  : "fromlf",
    "In From RF"  : "fromrf",
    "L To R"      : "ltor",
    "R To L"      : "rtol",  
    "Out To CF"   : "tocf",  
    "Out To LF"   : "tolf",  
    "Out To RF"   : "torf",
    "None"        : "unknown",
    "Varies"      : "unknown",
    "Calm"        : "unknown"
}

def getGameWind(gameJson):
    gameData  = gameJson["gameData" ]
    weather   = gameData["weather"  ]
    wind      = weather ["wind"     ]

    speedAndDirection = wind.split(",")

    speed = speedAndDirection[0]
    speed = speed.replace("mph", "")
    speed = speed.lstrip()
    speed = speed.rstrip()
    
    direction = speedAndDirection[1]
    direction = direction.lstrip()
    direction = direction.rstrip()

    retroDirection = "unknown"

    if direction in mlbWindDirectionToRetroWindDirection:
        retroDirection = mlbWindDirectionToRetroWindDirection[direction]

    return speed,retroDirection

In [ ]:
def getGameDate(gameJson):
    gameData     = gameJson["gameData"]
    dateTime     = gameData["datetime"]
    officialDate = dateTime["officialDate"]

    return officialDate

In [ ]:
def getWalks(summary):
    walks = 0
            
    splitSummary = summary.split("|")
    
    if len(splitSummary) > 1:
        stats = splitSummary[1]
        stats = stats.lstrip()
        stats = stats.rstrip()
        
        refinedStats = stats.split(",")
        
        for refinedStat in refinedStats:
            refinedStat = refinedStat.lstrip()
            refinedStat = refinedStat.rstrip()
        
            countToStat = refinedStat.split(" ")
        
            if len(countToStat) == 1:
                if refinedStat == "BB":
                    walks = 1
            else:
                if countToStat[1] == "BB":
                    walks = int(countToStat[0])

    return walks

In [ ]:
class BatterGame:
    def __init__(self, gid, bid, team, lineupPosition, positionOrder, stattype, plateAppearences,
                 atBats, runs, hits, doubles, triples, homeRuns, rbis, sacHits, sacFlies, hitByPitches,
                 walks, intentionalWalks, strikeOuts, stolenBases, caughtStealing, groundIntoDoublePlay,
                 catcherInterference, reachedOnError, designatedHitter, pinchHitter, pinchRunner, date,
                 number, site, visHome, opponent, win, loss, tie, gameType):
        self.gid = gid
        self.bid = bid
        self.team = team
        self.lineupPosition = lineupPosition
        self.positionOrder = positionOrder
        self.stattype = stattype
        self.plateAppearences = plateAppearences
        self.atBats = atBats
        self.runs = runs
        self.hits = hits
        self.doubles = doubles
        self.triples = triples
        self.homeRuns = homeRuns
        self.rbis = rbis
        self.sacHits = sacHits
        self.sacFlies = sacFlies
        self.hitByPitches = hitByPitches
        self.walks = walks
        self.intentionalWalks = intentionalWalks
        self.strikeOuts = strikeOuts
        self.stolenBases = stolenBases
        self.caughtStealing = caughtStealing
        self.groundIntoDoublePlay = groundIntoDoublePlay
        self.catcherInterference = catcherInterference
        self.reachedOnError = reachedOnError
        self.designatedHitter = designatedHitter
        self.pinchHitter = pinchHitter
        self.pinchRunner = pinchRunner
        self.date = date
        self.number = number
        self.site = site
        self.visHome = visHome
        self.opponent = opponent
        self.win = win
        self.loss = loss
        self.tie = tie
        self.gameType = gameType

    def getInsertQuery(self):
        #query  = "INSERT INTO batters (gid, id, team, b_lp, b_seq, stattype, b_pa, b_ab, b_r, b_h, b_d, b_t, b_hr, b_rbi, "
        #query += "b_sh, b_sf, b_hbp, b_w, b_iw, b_k, b_sb, b_cs, b_gdp, b_xi, b_roe, dh, ph, pr, date, number, site, vishome, "
        #query += "opp, win, loss, tie, gametype, box, pbp, manually_generated) VALUES"
        query  = f"(\"{self.gid}\", \"{self.bid}\", \"{self.team}\", \"{self.lineupPosition}\", \"{self.positionOrder}\", "
        query += f"\"{self.stattype}\", \"{self.plateAppearences}\", \"{self.atBats}\", \"{self.runs}\", \"{self.hits}\", "
        query += f"\"{self.doubles}\", \"{self.triples}\", \"{self.homeRuns}\", \"{self.rbis}\", \"{self.sacHits}\", \"{self.sacFlies}\", "
        query += f"\"{self.hitByPitches}\", \"{self.walks}\", \"{self.intentionalWalks}\", \"{self.strikeOuts}\", \"{self.stolenBases}\", "
        query += f"\"{self.caughtStealing}\", \"{self.groundIntoDoublePlay}\", \"{self.catcherInterference}\", \"{self.reachedOnError}\", "
        query += f"\"{self.designatedHitter}\", \"{self.pinchHitter}\", \"{self.pinchRunner}\", \"{self.date}\", \"{self.number}\", "
        query += f"\"{self.site}\", \"{self.visHome}\", \"{self.opponent}\", \"{self.win}\", \"{self.loss}\", \"{self.tie}\", \"{self.gameType}\", "
        query += "\"y\", \"y\", 1)"

        return query

    def __eq__(self, other):
        if not isinstance(other, BatterGame):
            return False

        return self.gid == other.gid and self.bid == other.bid and self.team == other.team and self.lineupPosition == other.lineupPosition and self.positionOrder == other.positionOrder and self.stattype == other.stattype and self.plateAppearences == other.plateAppearences and self.atBats == other.atBats and self.runs == other.runs and self.hits == other.hits and self.doubles == other.doubles and self.triples == other.triples and self.homeRuns == other.homeRuns and self.rbis == other.rbis and self.sacHits == other.sacHits and self.sacFlies == other.sacFlies and self.hitByPitches == other.hitByPitches and self.walks == other.walks and self.intentionalWalks == other.intentionalWalks and self.strikeOuts == other.strikeOuts and self.stolenBases == other.stolenBases and self.caughtStealing == other.caughtStealing and self.groundIntoDoublePlay == other.groundIntoDoublePlay and self.catcherInterference == other.catcherInterference and self.reachedOnError == other.reachedOnError and self.designatedHitter == other.designatedHitter and self.pinchHitter == other.pinchHitter and self.pinchRunner == other.pinchRunner and self.date == other.date and self.number == other.number and self.site == other.site and self.visHome == other.visHome and self.opponent == other.opponent and self.win == other.win and self.loss == other.loss and self.tie == other.tie and self.gameType == other.gameType

    def printGame(self):
        print(f"{self.gid}, {self.bid}, {self.team}, {self.lineupPosition}, {self.positionOrder}, {self.stattype}, {self.plateAppearences}, {self.atBats}, {self.runs}, {self.hits}, {self.doubles}, {self.triples}, {self.homeRuns}, {self.rbis}, {self.sacHits}, {self.sacFlies}, {self.hitByPitches}, {self.walks}, {self.intentionalWalks}, {self.strikeOuts}, {self.stolenBases}, {self.caughtStealing}, {self.groundIntoDoublePlay}, {self.catcherInterference}, {self.reachedOnError}, {self.designatedHitter}, {self.pinchHitter}, {self.date}, {self.number}, {self.site}, {self.visHome}, {self.opponent}, {self.win}, {self.loss}, {self.tie}, {self.gameType}")

In [ ]:
def getRetroBatting(player, gameId, gameDate, gameNumber, gameSite, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam, isHome):
    mlbId = player["person"]["id"]
    
    stats         = player["stats"   ]
    battingStats  = stats ["batting" ]
    pitchingStats = stats ["pitching"]
    fieldingStats = stats ["fielding"]

    visHome = ""
    
    if isHome:
        visHome = "h"
    else:
        visHome = "v"
    
    win  = 0
    loss = 0
    tie  = 0
    
    if homeTeamAbbr == winningTeam and isHome:
        win = 1
    elif homeTeamAbbr == winningTeam and isHome == False:
        loss = 1
    elif awayTeamAbbr == winningTeam and isHome == False:
        win = 1
    else:
        loss = 1

    teamAbbr = ""

    if isHome:
        teamAbbr = homeTeamAbbr
    else:
        teamAbbr = awayTeamAbbr

    oppTeam = ""

    if isHome:
        oppTeam = awayTeamAbbr
    else:
        oppTeam = homeTeamAbbr
    
    #ADD all 0's for batting if there are pitching/fielding stats present, if none do not add them
    if len(battingStats) > 0:
        retroId = getRetroIdFromMlbId(mlbId)
        
        lineupPositionCode = int(player["battingOrder"])
        lineupPosition     = int(lineupPositionCode/100)
        battingSequence    = (lineupPositionCode % 100) + 1
        
        positions = player["allPositions"]
        
        designatedHitter = ""
        pinchHitter      = ""
        pinchRunner      = ""
        
        for position in positions:
            positionCode = position["code"]
        
            if positionCode == "10":
                designatedHitter = "1"
            elif positionCode == "11":
                pinchHitter = "1"
            elif positionCode == "12":
                pinchRunner = "1"
        
        summary = battingStats["summary"]

        walks = getWalks(summary)
        
        atBats           = int(battingStats["atBats"]          )
        intentionalWalks = int(battingStats["intentionalWalks"])
        hitByPitches     = int(battingStats["hitByPitch"]      )
        sacHits          = int(battingStats["sacBunts"]        )
        sacFlies         = int(battingStats["sacFlies"]        )
        
        plateAppearences = atBats + walks + hitByPitches + sacHits + sacFlies
    
        atBats               = int(battingStats["atBats"])
        runs                 = int(battingStats["runs"])
        hits                 = int(battingStats["hits"])
        doubles              = int(battingStats["doubles"])
        triples              = int(battingStats["triples"])
        homeRuns             = int(battingStats["homeRuns"])
        rbis                 = int(battingStats["rbi"])
        sacHits              = int(battingStats["sacBunts"])
        sacFlies             = int(battingStats["sacFlies"])
        hitByPitch           = int(battingStats["hitByPitch"])
        intentionalWalks     = int(battingStats["intentionalWalks"])
        strikeOuts           = int(battingStats["strikeOuts"])
        stolenBases          = int(battingStats["stolenBases"])
        caughtStealing       = int(battingStats["caughtStealing"])
        groundIntoDoublePlay = int(battingStats["groundIntoDoublePlay"])
        catcherInterference  = int(battingStats["catchersInterference"])
        
        curGame = BatterGame(gameId, retroId, teamAbbr, lineupPosition, battingSequence, "value", plateAppearences, atBats,
                       runs, hits, doubles, triples, homeRuns, rbis, sacHits, sacFlies, hitByPitch, walks, intentionalWalks,
                       strikeOuts, stolenBases, caughtStealing, groundIntoDoublePlay, catcherInterference, -1, designatedHitter,
                        pinchHitter, pinchRunner, gameDate, gameNumber, gameSite, visHome, oppTeam, win, loss, tie, gameType)
        

        return curGame

    elif len(pitchingStats) > 0 or len(fieldingStats) > 0:
        retroId = getRetroIdFromMlbId(mlbId)
        
        curGame = BatterGame(gameId, retroId, teamAbbr, "", "", "value", 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                             0, 0, 0, 0, 0, 0, 0, 0, -1, "", "", "", gameDate, gameNumber, gameSite, visHome, awayTeamAbbr, win,
                             loss, tie, gameType)
        
        return curGame
    else:
        return None

In [ ]:
def getWalksForPitcher(summary):
    walks = 0
            
    refinedStats = summary.split(",")
    
    if len(refinedStats) > 1:
        for refinedStat in refinedStats:
            refinedStat = refinedStat.lstrip()
            refinedStat = refinedStat.rstrip()
        
            countToStat = refinedStat.split(" ")
        
            if len(countToStat) == 1:
                if refinedStat == "BB":
                    walks = 1
            else:
                if countToStat[1] == "BB":
                    walks = int(countToStat[0])

    return walks

In [ ]:
def getPitchingSequence(gameJson, isHome):
    if isHome:
        return gameJson["liveData"]["boxscore"]["teams"]["home"]["pitchers"]
    else:
        return gameJson["liveData"]["boxscore"]["teams"]["away"]["pitchers"]

In [ ]:
class PitcherGame:
    def __init__(self, gid, pid, team, pitchingSequence, statType, outs, noOuts, battersFaced, hits, doubles, triples, homeRuns, runs, earnedRuns,
                 walks, intentionalWalks, strikeOuts, hitByPitches, wildPitches, balks, sacHits, sacFlies, stolenBases, caughtStealing, passedBalls,
                 winningPitcher, losingPitcher, savingPitcher, gameStarted, gameFinished, completeGame, date, number, site, visHome, opp, win, loss,
                 tie, gameType):
        self.gid = gid
        self.pid = pid
        self.team = team
        self.pitchingSequence = pitchingSequence
        self.statType = statType
        self.outs = outs
        self.noOuts = noOuts
        self.battersFaced = battersFaced
        self.hits = hits
        self.doubles = doubles
        self.triples = triples
        self.homeRuns = homeRuns
        self.runs = runs
        self.earnedRuns = earnedRuns
        self.walks = walks
        self.intentionalWalks = intentionalWalks
        self.strikeOuts = strikeOuts
        self.hitByPitches = hitByPitches
        self.wildPitches = wildPitches
        self.balks = balks
        self.sacHits = sacHits
        self.sacFlies = sacFlies
        self.stolenBases = stolenBases
        self.caughtStealing = caughtStealing
        self.passedBalls = passedBalls
        self.winningPitcher = winningPitcher
        self.losingPitcher = losingPitcher
        self.savingPitcher = savingPitcher
        self.gameStarted = gameStarted
        self.gameFinished = gameFinished
        self.completeGame = completeGame
        self.date = date
        self.number = number
        self.site = site
        self.visHome = visHome
        self.opp = opp
        self.win = win
        self.loss = loss
        self.tie = tie
        self.gameType = gameType

    def getInsertQuery(self):
        #query  = "INSERT INTO pitching (gid, id, team, p_seq, stattype, p_ipouts, p_noout, p_bfp, p_h, p_d, p_t, "
        #query += "p_hr, p_r, p_er, p_w, p_iw, p_k, p_hbp, p_wp, p_bk, p_sh, p_sf, p_sb, p_cs, p_pb, wp, lp, "
        #query += "save, p_gs, p_gf, p_cg, date, number, site, vishome, opp, win, loss, tie, gametype, box, pbp, manually_generated) VALUES"
        query  = f"(\"{self.gid}\", \"{self.pid}\", \"{self.team}\", \"{self.pitchingSequence}\", \"{self.statType}\", \"{self.outs}\", "
        query += f"\"{self.noOuts}\", \"{self.battersFaced}\", \"{self.hits}\", \"{self.doubles}\", \"{self.triples}\", "
        query += f"\"{self.homeRuns}\", \"{self.runs}\", \"{self.earnedRuns}\", \"{self.walks}\", \"{self.intentionalWalks}\", "
        query += f"\"{self.strikeOuts}\", \"{self.hitByPitches}\", \"{self.wildPitches}\", \"{self.balks}\", \"{self.sacHits}\", "
        query += f"\"{self.sacFlies}\", \"{self.stolenBases}\", \"{self.caughtStealing}\", \"{self.passedBalls}\", "
        query += f"\"{self.winningPitcher}\", \"{self.losingPitcher}\", \"{self.savingPitcher}\", \"{self.gameStarted}\", "
        query += f"\"{self.gameFinished}\", \"{self.completeGame}\", \"{self.date}\", \"{self.number}\", \"{self.site}\", "
        query += f"\"{self.visHome}\", \"{self.opp}\", \"{self.win}\", \"{self.loss}\", \"{self.tie}\", \"{self.gameType}\", "
        query += "\"y\", \"y\", 1)"
        
        return query

        def __eq__(self, other):
            if not isinstance(other, PitcherGame):
                return False

            return self.gid == other.gid and self.pid == other.pid and self.team == other.team and self.pitchingSequence == other.pitchingSequence and self.statType == other.statType and self.outs == other.outs and self.noOuts == other.noOuts and self.battersFaced == other.battersFaced and self.hits == other.hits and self.doubles == other.doubles and self.triples == other.triples and self.homeRuns == other.homeRuns and self.runs == other.runs and self.earnedRuns == other.earnedRuns and self.walks == other.walks and self.intentionalWalk == other.intentionalWalks and self.strikeOuts == other.strikeOuts and self.hitByPitches == other.hitByPitches and self.wildPitches == other.wildPitches and self.balks == other.balks and self.sacHits == other.sacHits and self.sacFlies == other.sacFlies and self.stolenBases == other.stolenBases and self.caughtStealing == other.caughtStealing and self.passedBalls == other.passedBalls and self.winningPitcher == other.winningPitcher and self.losingPitcher == other.losingPitcher and self.savingPitcher == other.savingPitcher and self.gameStarted == other.gameStarted and self.gameFinished == other.gameFinished and self.completeGame == other.completeGame and self.date == other.date and self.number == other.number and self.site == other.site and self.visHome == other.visHome and self.opp == other.opp and self.win == other.win and self.loss == other.loss and self.tie == other.tie and self.gameType == other.gameType

In [ ]:
def getRetroPitching(player, gameJson, gameId, gameDate, gameNumber, gameSite, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam, isHome):
    stats         = player["stats"   ]
    pitchingStats = stats ["pitching"]

    pitchingSequence = getPitchingSequence(gameJson, isHome)

    if len(pitchingStats) > 0:
        mlbId = player["person"]["id"]

        retroId = getRetroIdFromMlbId(mlbId)

        visHome = ""
    
        if isHome:
            visHome = "h"
        else:
            visHome = "v"

        teamAbbr = ""
    
        if isHome:
            teamAbbr = homeTeamAbbr
        else:
            teamAbbr = awayTeamAbbr
    
        oppTeam = ""
    
        if isHome:
            oppTeam = awayTeamAbbr
        else:
            oppTeam = homeTeamAbbr
        
        win  = 0
        loss = 0
        tie  = 0
        
        if homeTeamAbbr == winningTeam and isHome:
            win = 1
        elif homeTeamAbbr == winningTeam and isHome == False:
            loss = 1
        elif awayTeamAbbr == winningTeam and isHome == False:
            win = 1
        else:
            loss = 1

        walks = getWalksForPitcher(pitchingStats["summary"])
                                              
        pitchingSequence = pitchingSequence.index(mlbId) + 1
        outsRecorded     = pitchingStats["outs"            ]
        #TODO: batters faced by in any innings in which no outs were recorded
        battersFaced     = pitchingStats["battersFaced"    ]
        hits             = pitchingStats["hits"            ]
        doubles          = pitchingStats["doubles"         ]
        triples          = pitchingStats["triples"         ]
        homeRuns         = pitchingStats["homeRuns"        ]
        runs             = pitchingStats["runs"            ]
        earnedRuns       = pitchingStats["earnedRuns"      ]
        intentionalWalks = pitchingStats["intentionalWalks"]
        strikeOuts       = pitchingStats["strikeOuts"      ]
        hitByPitches     = pitchingStats["hitByPitch"      ]
        wildPitches      = pitchingStats["wildPitches"     ]
        balks            = pitchingStats["balks"           ]
        sacrificeHits    = pitchingStats["sacBunts"        ]
        sacrificeFlies   = pitchingStats["sacFlies"        ]
        stolenBases      = pitchingStats["stolenBases"     ]
        caughtStealing   = pitchingStats["caughtStealing"  ]
        passedBalls      = pitchingStats["passedBall"      ]
        winningPitcher   = pitchingStats["wins"            ]
        losingPitcher    = pitchingStats["losses"          ]
        save             = pitchingStats["saves"           ]
        gameStarted      = pitchingStats["gamesStarted"    ]
        gameFinished     = pitchingStats["gamesFinished"   ]
        completeGame     = pitchingStats["completeGames"   ]

        # (self, gid, pid, team, pitchingSequence, statType, outs, noOuts, battersFaced, hits, double, triples, homeRuns, runs, earnedRuns,
        #          walks, intentionalWalks, strikeOuts, hitByPitches, wildPitches, balks, sacHits, sacFlies, stolenBases, caughtStealing, passedBalls,
        #          winningPitcher, losingPitcher, savingPitcher, gameStarted, gameFinished, completeGame, date, number, site, visHome, opp, win, loss,
        #          tie, gameType)
        
        curPitcher = PitcherGame(gameId, retroId, teamAbbr, pitchingSequence, "value", outsRecorded, -1, battersFaced, hits, doubles, triples, homeRuns, runs,
                                 earnedRuns, walks, intentionalWalks, strikeOuts, hitByPitches, wildPitches, balks, sacrificeHits, sacrificeFlies,
                                 stolenBases, caughtStealing, passedBalls, winningPitcher, losingPitcher, save, gameStarted, gameFinished, completeGame,
                                 gameDate, gameNumber, gameSite, visHome, oppTeam, win, loss, tie, gameType)

        return curPitcher

In [ ]:
def getBattingAndPitchingStats(gameJson, gameId, gameDate, gameNumber, gameSite, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam):
    homePlayers = gameJson["liveData"]["boxscore"]["teams"]["home"]["players"]
    awayPlayers = gameJson["liveData"]["boxscore"]["teams"]["away"]["players"]

    batters  = []
    pitchers = []
    for p in homePlayers:
        player = homePlayers[p]

        curBatter = getRetroBatting (player, gameId, gameDate, gameNumber, gameSite, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam, True)

        if curBatter != None:
            batters.append(curBatter)
        
        curPitcher = getRetroPitching(player, gameJson, gameId, gameDate, gameNumber, gameSite, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam, True)

        if curPitcher != None:
            pitchers.append(curPitcher)
            
    for p in awayPlayers:
        player = awayPlayers[p]

        curBatter = getRetroBatting (player, gameId, gameDate, gameNumber, gameSite, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam, False)

        if curBatter != None:
            batters.append(curBatter)
        
        curPitcher = getRetroPitching(player, gameJson, gameId, gameDate, gameNumber, gameSite, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam, False)

        if curPitcher != None:
            pitchers.append(curPitcher)

    return batters,pitchers

In [ ]:
def getSuspendedDate(gameJson):
    gameData = gameJson["gameData"]
    dateTime = gameData["datetime"]

    resumeDate = ""

    if "resumeDate" in dateTime:
        resumeDate = dateTime["resumeDate"]

    return resumeDate

In [ ]:
class Game:
    def __init__(self, gid, visTeam, homeTeam, site, date, number, startTime, dayNight, innings, tieBreaker, usedh, htbf, timeOfGame, attendance,
                 fieldCondition, precipitation, sky, temp, windDir, windSpeed, offScorer, forfeit, suspended, umpHome, umpFirstBase, umpSecondBase,
                 umpThirdBase, umpLeftField, umpRightField, winningPitcher, losingPitcher, savingPitcher, gameType, visRuns, homeRuns, winningTeam,
                 losingTeam, line, batteries, lineUps, boxScore, playByPlay, season):
        self.gid            = gid           
        self.visTeam        = visTeam       
        self.homeTeam       = homeTeam      
        self.site           = site          
        self.date           = date          
        self.number         = number        
        self.startTime      = startTime     
        self.dayNight       = dayNight      
        self.innings        = innings       
        self.tieBreaker     = tieBreaker    
        self.usedh          = usedh         
        self.htbf           = htbf          
        self.timeOfGame     = timeOfGame    
        self.attendance     = attendance    
        self.fieldCondition = fieldCondition
        self.precipitation  = precipitation 
        self.sky            = sky           
        self.temp           = temp          
        self.windDir        = windDir       
        self.windSpeed      = windSpeed     
        self.offScorer      = offScorer     
        self.forfeit        = forfeit       
        self.suspended      = suspended     
        self.umpHome        = umpHome       
        self.umpFirstBase   = umpFirstBase  
        self.umpSecondBase  = umpSecondBase 
        self.umpThirdBase   = umpThirdBase  
        self.umpLeftField   = umpLeftField  
        self.umpRightField  = umpRightField 
        self.winningPitcher = winningPitcher
        self.losingPitcher  = losingPitcher 
        self.savingPitcher  = savingPitcher 
        self.gameType       = gameType      
        self.visRuns        = visRuns       
        self.homeRuns       = homeRuns      
        self.winningTeam    = winningTeam   
        self.losingTeam     = losingTeam    
        self.line           = line          
        self.batteries      = batteries     
        self.lineUps        = lineUps       
        self.boxScore       = boxScore      
        self.playByPlay     = playByPlay    
        self.season         = season

    def getInsertQuery(self):
        #query  = "INSERT into games (gid, visteam, hometeam, site, date, number, starttime, daynight, innings, "
        #query += "tiebreaker, usedh, htbf, timeofgame, attendance, fieldcond, precip, sky, temp, winddir, windspeed, "
        #query += "oscorer, forfeit, suspend, umphome, ump1b, ump2b, ump3b, umplf, umprf, wp, lp, save, gametype, "
        #query += "vruns, hruns, wteam, lteam, line, batteries, lineups, box, pbp, season, manually_generated) VALUES "
        query  = f"(\"{self.gid}\", \"{self.visTeam}\", \"{self.homeTeam}\", \"{self.site}\", \"{self.date}\", \"{self.number}\", "
        query += f"\"{self.startTime}\", \"{self.dayNight}\", \"{self.innings}\", \"{self.tieBreaker}\", \"{self.usedh}\", \"{self.htbf}\", "
        query += f"\"{self.timeOfGame}\", \"{self.attendance}\", \"{self.fieldCondition}\", \"{self.precipitation}\", \"{self.sky}\", "
        query += f"\"{self.temp}\", \"{self.windDir}\", \"{self.windSpeed}\", \"{self.offScorer}\", \"{self.forfeit}\", \"{self.suspended}\", "
        query += f"\"{self.umpHome}\", \"{self.umpFirstBase}\", \"{self.umpSecondBase}\", \"{self.umpThirdBase}\", \"{self.umpLeftField}\", "
        query += f"\"{self.umpRightField}\", \"{self.winningPitcher}\", \"{self.losingPitcher}\", \"{self.savingPitcher}\", \"{self.gameType}\", "
        query += f"\"{self.visRuns}\", \"{self.homeRuns}\", \"{self.winningTeam}\", \"{self.losingTeam}\", \"{self.line}\", \"{self.batteries}\", "
        query += f"\"{self.lineUps}\", \"{self.boxScore}\", \"{self.playByPlay}\", \"{self.season}\", 1)"

        return query

    def __eq__(self, other):
            if not isinstance(other, Game):
                return False

            return self.gid == gid and self.visTeam == visTeam and self.homeTeam == homeTeam and self.site == site and self.date == date and self.number == number and self.startTime == startTime and self.dayNight == dayNight and self.innings == innings and self.tieBreaker == tieBreaker and self.usedh == usedh and self.htbf == htbf and self.timeOfGame == timeOfGame and self.attendance == attendance and self.fieldCondition == fieldCondition and self.precipitation == precipitation and self.sky == sky and self.temp == temp and self.windDir == windDir and self.windSpeed == windSpeed and self.offScorer == offScorer and self.forfeit == forfeit and self.suspended == suspended and self.umpHome == umpHome and self.umpFirstBase == umpFirstBase and self.umpSecondBase == umpSecondBase and self.umpThirdBase == umpThirdBase and self.umpLeftField == umpLeftField and self.umpRightField == umpRightField and self.winningPitcher == winningPitcher and self.losingPitcher == losingPitcher and self.savingPitcher == savingPitcher and self.gameType == gameType and self.visRuns == visRuns and self.homeRuns == homeRuns and self.winningTeam == winningTeam and self.losingTeam == losingTeam and self.line == line and self.batteries == batteries and self.lineUps == lineUps and self.boxScore == boxScore and self.playByPlay == playByPlay and self.season == season

In [ ]:
def getGameStats(gameJson):
    homeTeam = getTeamAbbreviation(gameJson, True)
    visTeam = getTeamAbbreviation(gameJson, False)
    site = getVenue(gameJson)
    date = getGameDate(gameJson)
    number = getGameNumber(gameJson)
    startTime = getGameTime(gameJson)
    dayNight = getDayOrNight(gameJson)
    innings = getInningsPlayed(gameJson)
    tieBreaker = getTieBreaker(gameJson)
    usedh = "true" #getUsedDh(gameJson)
    htbf = ""
    timeOfGame = getGameDuration(gameJson)
    attendance = getGameAttendance(gameJson)
    fieldCondition = "unknown"
    precipitation = getGamePrecipitation(gameJson)
    sky = getGameCondition(gameJson)
    temp = getGameTemp(gameJson)
    windSpeed,windDir = getGameWind(gameJson)
    #TODO: Add backup if offscorer isnt present in retrosheet
    offScorer = getOfficialScorer(gameJson)
    forfeit = ""
    suspended = getSuspendedDate(gameJson)
    gameId = getGameId(gameJson, homeTeam, number)

    #TODO: Add backup if offscorer isnt present in retrosheet
    umpires = getUmpires(gameJson)

    umpHome       = "(none)"
    umpFirstBase  = "(none)"
    umpSecondBase = "(none)"
    umpThirdBase  = "(none)"
    umpLeftField  = "(none)"
    umpRightField = "(none)"

    if "Home Plate" in umpires:
        umpHome = umpires["Home Plate"]
    if "First Base" in umpires:
        umpFirstBase = umpires["First Base"]
    if "Second Base" in umpires:
        umpSecondBase = umpires["Second Base"]
    if "Third Base" in umpires:
        umpThirdBase = umpires["Third Base"]
    if "Left Field" in umpires:
        umpLeftField = umpires["Left Field"]
    if "Right Field" in umpires:
        umpRightField = umpires["Right Field"]

    #TODO: Add backup if offscorer isnt present in retrosheet
    winningPitcherId, losingPitcherId, savingPitcherId = getWinningLosingAndSavingPitcher(gameJson)

    gameType = getGameType(gameJson)

    visRuns  = gameJson["liveData"]["boxscore"]["teams"]["away"]["teamStats"]["batting"]["runs"]
    homeRuns = gameJson["liveData"]["boxscore"]["teams"]["home"]["teamStats"]["batting"]["runs"]

    winningTeam = getWinningTeam(gameJson)
    losingTeam  = getLosingTeam (gameJson)

    line      = "y"
    batteries = "both"
    lineUps   = "y"
    box       = "y"
    pbp       = "y"

    gameDateSplit = datetime.strptime(date, "%Y-%m-%d")
    season        = gameDateSplit.year

    curGame = Game(gameId, visTeam, homeTeam, site, date, number, startTime, dayNight, innings, tieBreaker, usedh, htbf,
                    timeOfGame, attendance, fieldCondition, precipitation, sky, temp, windDir, windSpeed, offScorer, forfeit,
                    suspended, umpHome, umpFirstBase, umpSecondBase, umpThirdBase, umpLeftField, umpRightField, winningPitcherId,
                    losingPitcherId, savingPitcherId, gameType, visRuns, homeRuns, winningTeam, losingTeam, line, batteries,
                    lineUps, box, pbp, season)

    return curGame

In [ ]:
class Play:
    def __init__(self, gid, inning, topOrBot, visHome, gameSite, batTeam, pitTeam, batter,
                       pitcher, batHand, pitHand, count, numP, plateAppearence,
                       atBat, single, double, triple, homeRun, sacHit, sacFly, hitByPitch, walk,
                       intentionalWalk, strikeOut, catcherInterference, runs,
                       rbis, earnedRuns, teamUnearnedRuns, date, gameType, atBatIndex):
        self.gid                   = gid                  
        self.inning                = inning               
        self.topOrBot              = topOrBot              
        self.visHome               = visHome             
        self.gameSite              = gameSite             
        self.batTeam               = batTeam              
        self.pitTeam               = pitTeam              
        self.batter                = batter               
        self.pitcher               = pitcher              
        self.batHand               = batHand              
        self.pitHand               = pitHand              
        self.count                 = count                
        self.numP                  = numP                 
        self.plateAppearence       = plateAppearence      
        self.atBat                 = atBat                
        self.single                = single               
        self.double                = double               
        self.triple                = triple               
        self.homeRun               = homeRun              
        self.sacHit                = sacHit               
        self.sacFly                = sacFly               
        self.hitByPitch            = hitByPitch           
        self.walk                  = walk         
        self.intentionalWalk       = intentionalWalk      
        self.strikeOut             = strikeOut            
        self.catcherInterference   = catcherInterference  
        self.runs                  = runs                 
        self.rbis                  = rbis                 
        self.earnedRuns            = earnedRuns           
        self.teamUnearnedRuns      = teamUnearnedRuns     
        self.date                  = date                 
        self.gameType              = gameType
        self.atBatIndex            = atBatIndex
        self.pitchLogs             = []

    def getInsertQuery(self):
        #query  = "INSERT INTO plays (gid, event, inning, top_bot, vis_home, ballpark, batteam, pitteam, "
        #query += "batter, pitcher, lp, bat_f, bathand, pithand, count, pitches, nump, pa, ab, single, "
        #query += "double, triple, hr, sh, sf, hbp, walk, iw, k, xi, oth, othout, noout, bip, bunt, ground, fly, line, "
        #query += "gdp, othdp, tp, wp, pb, bk, oa, di, sb2, sb3, sbh, cs2, cs3, csh, pko1, pko2, pko3, k_safe, e1, e2, "
        #query += "e3, e4, e5, e6, e7, e8, e9, outs_pre, outs_post, br1_pre, br2_pre, br3_pre, br1_post, br2_post, "
        #query += "br3_post, run_b, run1, run2, run3, prun1, prun2, prun3, runs, rbi, er, tur, f2, f3, f4, f5, f6, f7, f8, "
        #query += "f9, po0, po1, po2, po3, po4, po5, po6, po7, po8, po9, a1, a2, a3, a4, a5, a6, a7, a8, a9, batout1, batout2, "
        #query += " batout3, brout_b, brout1, brout2, brout3, firstf, loc, hittype, dpopp, pivot, pn, date, gametype, pbp, "
        #query += "manually_generated, atBatIndex) VALUES"
        query  = f"(\"{self.gid}\", \"\", \"{self.inning}\", \"{self.topOrBot}\", \"{self.visHome}\", \"{self.gameSite}\", "
        query += f"\"{self.batTeam}\", \"{self.pitTeam}\", \"{self.batter}\", \"{self.pitcher}\", \"\", \"\", \"{self.batHand}\", "
        query += f"\"{self.pitHand}\", \"{self.count}\", \"\", \"{self.numP}\", \"{self.plateAppearence}\", \"{self.atBat}\", "
        query += f"\"{self.single}\", \"{self.double}\", \"{self.triple}\", \"{self.homeRun}\", \"{self.sacHit}\", "
        query += f"\"{self.sacFly}\", \"{self.hitByPitch}\", \"{self.walk}\", \"{self.intentionalWalk}\", \"{self.strikeOut}\", "
        query += f"\"{self.catcherInterference}\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", "
        query += "\"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", "
        query += "\"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", "
        query += f"\"{self.runs}\", \"{self.rbis}\", \"{self.earnedRuns}\", \"{self.teamUnearnedRuns}\", \"\", \"\", \"\", \"\", \"\", "
        query += "\"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", "
        query += f"\"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"\", \"{self.date}\", \"{self.gameType}\", "
        query += f"\"y\", 1, \"{self.atBatIndex}\")"

        return query

In [ ]:
def getPlays(play, gid, date, gameType, gameSite, homeTeamAbbr, visTeamAbbr):
    about      = play["about"]
    matchup    = play["matchup"]
    count      = play["count"]
    pitchIndex = play["pitchIndex"]
    result     = play["result"]
    runners    = play["runners"]

    atBatIndex = about["atBatIndex"]
    
    inning = about["inning"     ]
    isTop  = about["isTopInning"]

    topOrBot = 0
    visHome  = 0
    batTeam  = visTeamAbbr
    pitTeam  = homeTeamAbbr

    if isTop == False:
        topOrBot = 1
        visHome  = 1
        batTeam  = homeTeamAbbr
        pitTeam  = visTeamAbbr

    batter  = matchup["batter" ]
    pitcher = matchup["pitcher"]

    batterMlbId  = batter ["id"]
    pitcherMlbId = pitcher["id"]

    batterRetroId  = getRetroIdFromMlbId(batterMlbId )
    pitcherRetroId = getRetroIdFromMlbId(pitcherMlbId)

    batSide = matchup["batSide"]
    batHand = batSide["code"]

    pitchHand     = matchup  ["pitchHand"]
    pitchHandCode = pitchHand["code"]

    balls   = count["balls"]
    strikes = count["strikes"]

    rbi   = result["rbi"]

    #ADD COUNT CODE
    curCount = "0"

    if balls == 0 and strikes > 0:
        curCount = str(strikes)
    elif balls > 0 and strikes == 0:
        curCount = str(balls)
    elif balls > 0 and strikes > 0:
        curCount = str(balls) + str(strikes)

    numPitches = len(pitchIndex)

    single                 = 0
    double                 = 0
    triple                 = 0
    homeRun                = 0
    sacHit                 = 0
    sacFly                 = 0
    hitByPitch             = 0
    walk                   = 0
    intentionalWalk        = 0
    strikeOut              = 0
    catcherInterference    = 0
    caughtStealing         = 0
    balk                   = 0
    wildPitch              = 0
    passedBall             = 0

    eventType = result["eventType"]

    match eventType:
        case "single":
            single = 1
        case "double":
            double = 1
        case "triple":
            triple = 1
        case "home_run":
            homeRun = 1
        case "sac_bunt" | "sac_bunt_double_play":
            sacHit = 1
        case "sac_fly" | "sac_fly_double_play":
            sacFly = 1
        case "hit_by_pitch":
            hitByPitch = 1
        case "walk":
            walk = 1
        case "intent_walk":
            intentionalWalk = 1
        case "strikeout" | "strikeout_double_play" | "strikeout_triple_play":
            strikeOut = 1
        case "catcher_interf":
            catcherInterference = 1
        case "pickoff_caught_stealing_2b" | "pickoff_caught_stealing_3b" | "pickoff_caught_stealing_home" | "caught_stealing_2b" | "caught_stealing_3b" | "caught_stealing_home":
            caughtStealing = 1
        case "forced_balk" | "balk":
            balk = 1
        case "wild_pitch":
            wildPitch = 1
        case "passed_ball":
            passedBall = 1

    plateAppearence = 0

    #pickoff_caught_stealing_2b, pickoff_caught_stealing_3b, pickoff_caught_stealing_home, caught_stealing_2b, caught_stealing_3b, caught_stealing_home, forced_balk, balk, wild_pitch, passed_ball
    if caughtStealing == 0 and balk == 0 and wildPitch == 0 and passedBall == 0:
        plateAppearence = 1

    atBat = 0
    
    if plateAppearence == 1 and walk != 1 and intentionalWalk != 1 and hitByPitch != 1 and sacHit != 1 and sacFly != 1 and catcherInterference != 1:
        atBat = 1
        
    earnedRuns           = 0
    teamUnearnedRuns     = 0
    runs                 = 0
    
    for curRunner in runners:
        details = curRunner["details" ]

        earned         = details["earned"        ]
        unearnedRuns   = details["teamUnearned"  ]
        isScoringEvent = details["isScoringEvent"]

        earnedRuns           = earnedRuns       + earned
        teamUnearnedRuns     = teamUnearnedRuns + unearnedRuns
        
        if isScoringEvent:
            runs = runs + 1

    curPlay = Play(gid, inning, topOrBot, visHome, gameSite, batTeam, pitTeam, batterRetroId,
                       pitcherRetroId, batHand, pitchHandCode, curCount, numPitches, plateAppearence,
                       atBat, single, double, triple, homeRun, sacHit, sacFly, hitByPitch, walk,
                       intentionalWalk, strikeOut, catcherInterference, runs,
                       rbi, earnedRuns, teamUnearnedRuns, date, gameType, atBatIndex)

    playEvents = play["playEvents"]

    for playEvent in playEvents:
        if "isPitch" in playEvent and playEvent["isPitch"] == True:
            pitchLog = getPitchLog(playEvent, gid, batterRetroId, pitcherRetroId, atBatIndex)

            curPlay.pitchLogs.append(pitchLog)

    return curPlay

In [ ]:
#plays = gameJson["liveData"]["plays"]["allPlays"]
#
#count = 0
#
#for play in plays:
#    count = count + 1
#    curPlay = getPlays(play, gameId, gameDate, gameType, venue, homeTeamAbbr, awayTeamAbbr)
#
#    #if curPlay.atBatIndex == 59:
#    #    curPlay.getInsertQuery()
#
#print(count)

In [ ]:
class PitchLog:
    def __init__(self, gid, bid, pid, atBatIndex, call, ballColor, trailColor, inPlay, isStrike, isBall, pitchType, isOut,
                 hasReview, balls, strikes, outs, pitchStartSpeed, pitchEndSpeed, strikeZoneTop, strikeZoneBottom, aY, aZ,
                 pfxX, pfxZ, pX, pZ, vX0, vY0, vZ0, x, y, x0, y0, z0, aX, breakAngle, breakLength, breakY, breakVertical,
                 breakVerticalInduced, breakHorizontal, spinRate, spinDirection, pitchNumber, hitLaunchAngle, hitLaunchSpeed,
                 hitTotalDistance, hitTrajectory, hitHardness, hitLocation, hitCoordX, hitCoordY):
        self.gid                  = gid                
        self.bid                  = bid                
        self.pid                  = pid                
        self.atBatIndex           = atBatIndex         
        self.call                 = call               
        self.ballColor            = ballColor          
        self.trailColor           = trailColor         
        self.inPlay               = inPlay             
        self.isStrike             = isStrike           
        self.isBall               = isBall             
        self.pitchType            = pitchType          
        self.isOut                = isOut              
        self.hasReview            = hasReview          
        self.balls                = balls              
        self.strikes              = strikes            
        self.outs                 = outs               
        self.pitchStartSpeed      = pitchStartSpeed    
        self.pitchEndSpeed        = pitchEndSpeed      
        self.strikeZoneTop        = strikeZoneTop      
        self.strikeZoneBottom     = strikeZoneBottom   
        self.aY                   = aY                 
        self.aZ                   = aZ                 
        self.pfxX                 = pfxX               
        self.pfxZ                 = pfxZ               
        self.pX                   = pX                 
        self.pZ                   = pZ                 
        self.vX0                  = vX0                
        self.vY0                  = vY0                
        self.vZ0                  = vZ0                
        self.x                    = x                  
        self.y                    = y                  
        self.x0                   = x0                 
        self.y0                   = y0                 
        self.z0                   = z0                 
        self.aX                   = aX                 
        self.breakAngle           = breakAngle         
        self.breakLength          = breakLength        
        self.breakY               = breakY             
        self.breakVertical        = breakVertical      
        self.breakVerticalInduced = breakVerticalInduced
        self.breakHorizontal      = breakHorizontal    
        self.spinRate             = spinRate           
        self.spinDirection        = spinDirection      
        self.pitchNumber          = pitchNumber        
        self.hitLaunchAngle       = hitLaunchAngle     
        self.hitLaunchSpeed       = hitLaunchSpeed     
        self.hitTotalDistance     = hitTotalDistance   
        self.hitTrajectory        = hitTrajectory      
        self.hitHardness          = hitHardness        
        self.hitLocation          = hitLocation        
        self.hitCoordX            = hitCoordX          
        self.hitCoordY            = hitCoordY          

    def getInsertQuery(self):
        #query  = "INSERT INTO pitch_log (gid, bid, pid, atBatIndex, call, ballColor, trailColor, inPlay, isStrike, isBall, pitchType, "
        #query += "isOut, hasReview, balls, strikes, outs, pitchStartSpeed, pitchEndSpeed, strikeZoneTop, strikeZoneBottom, aY, "
        #query += "aZ, pfxX, pfxZ, pX, pZ, vX0, vY0, vZ0, x, y, x0, y0, z0, aX, breakAngle, breakLength, breakY, breakVertical, "
        #query += "breakVerticalInduced, breakHorizontal, spinRate, spinDirection, pitchNumber, hitLaunchAngle, hitLaunchSpeed, "
        #query += "hitTotalDistance, hitTrajectory, hitHardness, hitLocation, hitCoordX, hitCoordY) VALUES"
        query  = f"(\"{self.gid}\", \"{self.bid}\", \"{self.pid}\", {self.atBatIndex}, \"{self.call}\", \"{self.ballColor}\", \"{self.trailColor}\", "
        query += f"\"{self.inPlay}\", \"{self.isStrike}\", \"{self.isBall}\", \"{self.pitchType}\", "
        query += f"\"{self.isOut}\", \"{self.hasReview}\", {self.balls}, {self.strikes}, {self.outs}, {self.pitchStartSpeed}, {self.pitchEndSpeed}, {self.strikeZoneTop}, "
        query += f"{self.strikeZoneBottom}, {self.aY}, {self.aZ}, {self.pfxX}, {self.pfxZ}, {self.pX}, {self.pZ}, {self.vX0}, {self.vY0}, "
        query += f"{self.vZ0}, {self.x}, {self.y}, {self.x0}, {self.y0}, {self.z0}, {self.aX}, {self.breakAngle}, {self.breakLength}, {self.breakY}, "
        query += f"{self.breakVertical}, {self.breakVerticalInduced}, {self.breakHorizontal}, {self.spinRate}, {self.spinDirection}, {self.pitchNumber}, "
        query += f"{self.hitLaunchAngle}, {self.hitLaunchSpeed}, {self.hitTotalDistance}, \"{self.hitTrajectory}\", \"{self.hitHardness}\", {self.hitLocation}, "
        query += f"{self.hitCoordX}, {self.hitCoordY})"

        return query

In [ ]:
def getDataFromMap(dataMap, key, defaultValue):
    if key in dataMap:
        return dataMap[key]
    else:
        return defaultValue

In [ ]:
def getPitchLog(playEvent, gameId, pitcher, batter, atBatIndex):
    pitchNumber = playEvent["pitchNumber"]
    
    details     = playEvent["details"]
    ballColor   = getDataFromMap(details, "ballColor", "")
    trailColor  = getDataFromMap(details, "trailColor", "")
    inPlay      = str(getDataFromMap(details, "isInPlay", "false")).lower()
    isStrike    = str(getDataFromMap(details, "isStrike", "")).lower()
    isBall      = str(getDataFromMap(details, "isBall", "")).lower()
    isOut       = str(getDataFromMap(details, "isOut", "")).lower()
    hasReview   = str(getDataFromMap(details, "hasReview", "")).lower()

    call = details["call"]["code"]

    pitchType = ""

    if "type" in details:
        pType     = details["type"]
        pitchType = getDataFromMap(pType, "code", "")

    count   = playEvent["count"]
    balls   = getDataFromMap(count, "balls", -1)
    strikes = getDataFromMap(count, "strikes", -1)
    outs    = getDataFromMap(count, "outs", -1)

    pitchData        = playEvent["pitchData"]
    startSpeed       = getDataFromMap(pitchData, "startSpeed", -1.0)
    endSpeed         = getDataFromMap(pitchData, "endSpeed", -1.0)
    strikeZoneTop    = getDataFromMap(pitchData, "strikeZoneTop", -1.0)
    strikeZoneBottom = getDataFromMap(pitchData, "strikeZoneBottom", -1.0)

    coordinates = pitchData["coordinates"]
    aY          = getDataFromMap(coordinates, "aY", -1.0)
    aZ          = getDataFromMap(coordinates, "aZ", -1.0)
    pfxX        = getDataFromMap(coordinates, "pfxX", -1.0)
    pfxZ        = getDataFromMap(coordinates, "pfxZ", -1.0)
    pX          = getDataFromMap(coordinates, "pX", -1.0)
    pZ          = getDataFromMap(coordinates, "pZ", -1.0)
    vX0         = getDataFromMap(coordinates, "vX0", -1.0)
    vY0         = getDataFromMap(coordinates, "vY0", -1.0)
    vZ0         = getDataFromMap(coordinates, "vZ0", -1.0)
    x           = getDataFromMap(coordinates, "x", -1.0)
    y           = getDataFromMap(coordinates, "x", -1.0)
    x0          = getDataFromMap(coordinates, "x0", -1.0)
    y0          = getDataFromMap(coordinates, "y0", -1.0)
    z0          = getDataFromMap(coordinates, "z0", -1.0)
    aX          = getDataFromMap(coordinates, "aX", -1.0)

    breaks               = pitchData["breaks"]
    breakAngle           = getDataFromMap(breaks, "breakAngle", -1.0)
    breakLength          = getDataFromMap(breaks, "breakLength", -1.0)
    breakY               = getDataFromMap(breaks, "breakY", -1.0)
    breakVertical        = getDataFromMap(breaks, "breakVertical", -1.0)
    breakVerticalInduced = getDataFromMap(breaks, "breakVerticalInduced", -1.0)
    breakHorizontal      = getDataFromMap(breaks, "breakHorizontal", -1.0)
    spinRate             = getDataFromMap(breaks, "spinRate", -1)
    spinDirection        = getDataFromMap(breaks, "spinDirection", -1)

    launchSpeed   = -1.0
    launchAngle   = -1
    totalDistance = -1
    trajectory    = ""
    hardness      = ""
    location      = -1
    hitCoordX     = -1.0
    hitCoordY     = -1.0

    if "hitData" in playEvent:
        hitData       = playEvent["hitData"]
        trajectory    = getDataFromMap(hitData, "trajectory", "")
        hardness      = getDataFromMap(hitData, "hardness", "")
        location      = getDataFromMap(hitData, "location", -1)

        launchSpeed   = getDataFromMap(hitData, "launchSpeed", -1.0)
        launchAngle   = getDataFromMap(hitData, "launchAngle", -1)
        totalDistance = getDataFromMap(hitData, "totalDistance", -1)

        hitCoordX = -1.0
        hitCoordY = -1.0
        
        if "coordinates" in hitData:
            hitCoordinates = hitData["coordinates"]
        
            hitCoordX      = getDataFromMap(hitCoordinates, "coordX", -1.0)
            hitCoordY      = getDataFromMap(hitCoordinates, "coordY", -1.0)

    curPitchLog = PitchLog(gameId, batter, pitcher, atBatIndex, call, ballColor, trailColor, inPlay, isStrike, isBall,
                           pitchType, isOut, hasReview, balls, strikes, outs, startSpeed, endSpeed, strikeZoneTop,
                           strikeZoneBottom, aY, aZ, pfxX, pfxZ, pX, pZ, vX0, vY0, vZ0, x, y, x0, y0, z0, aX, breakAngle,
                           breakLength, breakY, breakVertical, breakVerticalInduced, breakHorizontal, spinRate, spinDirection,
                           pitchNumber, launchAngle, launchSpeed, totalDistance, trajectory, hardness, location, hitCoordX, hitCoordY)

    return curPitchLog

In [ ]:
def runInsertQuery(query):
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    cursor.execute(query)
    
    connection.commit()

    cursor    .close()
    connection.close()

In [ ]:
#3-18 -> 3->19
#3->27 -> 7-13
#7-18 -> 7-20
#2025-07-21 -> 2025-07-27

startDate = "2025-08-25"
endDate   = "2025-09-07"

gameScheduleRequest = f"https://statsapi.mlb.com/api/v1/schedule/games/?sportId=1&startDate={startDate}&endDate={endDate}"

gamesScheduleResponse = requests.get(gameScheduleRequest)
gameScheduleJson      = gamesScheduleResponse.json()

In [ ]:
# gameQueries     = []
# batterQueries   = []
# pitcherQueries  = []
# playQueries     = []
# pitchLogQueries = []

# dates = gameScheduleJson["dates"]

# i = 0

# for date in dates:
#     games = date["games"]

#     for game in games:
#         link        = game["link"]
#         curGameType = game["gameType"]
#         status      = game["status"]

#         codedGameState = status["codedGameState"]
        
#         if curGameType == 'R' and codedGameState == 'F':
#             print(link)
    
#             curGameUrl = "https://statsapi.mlb.com" + link
    
#             curGameResponse = requests.get(curGameUrl)
#             curGameJson     = curGameResponse.json()
    
#             curGame = getGameStats(curGameJson)
    
#             curGameInsertQuery = curGame.getInsertQuery()
            
#             gameQueries.append(curGameInsertQuery)
    
#             #runInsertQuery(curGameInsertQuery)
    
#             gameId       = curGame.gid
#             gameDate     = curGame.date
#             gameNumber   = curGame.number
#             venue        = curGame.site
#             gameType     = curGame.gameType
#             homeTeamAbbr = curGame.homeTeam
#             awayTeamAbbr = curGame.visTeam
#             winningTeam  = curGame.winningTeam
    
#             batters,pitchers = getBattingAndPitchingStats(curGameJson, gameId, gameDate, gameNumber, venue, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam)
    
#             for batter in batters:
#                 batterInsertQuery = batter.getInsertQuery()
#                 batterQueries.append(batterInsertQuery)
    
#                 #runInsertQuery(batterInsertQuery)
    
#             for pitcher in pitchers:
#                 pitcherInsertQuery = pitcher.getInsertQuery()
#                 pitcherQueries.append(pitcherInsertQuery)
    
#                 #runInsertQuery(pitcherInsertQuery)
            
#             #WE ARE HERE NOW
#             plays = curGameJson["liveData"]["plays"]["allPlays"]
    
#             for play in plays:
#                 play = getPlays(play, gameId, gameDate, gameType, venue, homeTeamAbbr, awayTeamAbbr)
    
#                 playInsertQuery = play.getInsertQuery()
#                 playQueries.append(playInsertQuery)
    
#                 #runInsertQuery(playInsertQuery)
    
#                 for pitchLog in play.pitchLogs:
#                     pitchLogInsertQuery = pitchLog.getInsertQuery()
#                     pitchLogQueries.append(pitchLogInsertQuery)
    
#                     #runInsertQuery(pitchLogInsertQuery)

#             i += 1
    
#             if i % 5 == 0:
#                 print("sleeping")
#                 time.sleep(5)

In [ ]:
gameLock    = threading.Lock()
batterLock  = threading.Lock()
pitcherLock = threading.Lock()
playLock    = threading.Lock()
pitchLock   = threading.Lock()

def getMlbDataFromGame(link):
    print(link)
    
    curGameUrl = "https://statsapi.mlb.com" + link
    
    curGameResponse = requests.get(curGameUrl)
    curGameJson     = curGameResponse.json()

    curGame = getGameStats(curGameJson)

    curGameInsertQuery = curGame.getInsertQuery()

    with gameLock:
        gameQueries.append(curGameInsertQuery)

    gameId       = curGame.gid
    gameDate     = curGame.date
    gameNumber   = curGame.number
    venue        = curGame.site
    gameType     = curGame.gameType
    homeTeamAbbr = curGame.homeTeam
    awayTeamAbbr = curGame.visTeam
    winningTeam  = curGame.winningTeam

    batters,pitchers = getBattingAndPitchingStats(curGameJson, gameId, gameDate, gameNumber, venue, gameType, homeTeamAbbr, awayTeamAbbr, winningTeam)

    for batter in batters:
        batterInsertQuery = batter.getInsertQuery()

        with batterLock:
            batterQueries.append(batterInsertQuery)

    for pitcher in pitchers:
        pitcherInsertQuery = pitcher.getInsertQuery()

        with pitcherLock:
            pitcherQueries.append(pitcherInsertQuery)

    plays = curGameJson["liveData"]["plays"]["allPlays"]
    
    for play in plays:
        play = getPlays(play, gameId, gameDate, gameType, venue, homeTeamAbbr, awayTeamAbbr)

        playInsertQuery = play.getInsertQuery()

        with playLock:
            playQueries.append(playInsertQuery)

        for pitchLog in play.pitchLogs:
            pitchLogInsertQuery = pitchLog.getInsertQuery()
            
            with pitchLock:
                pitchLogQueries.append(pitchLogInsertQuery)

In [ ]:
bulkGameInsertQuery  = "INSERT into games (gid, visteam, hometeam, site, date, number, starttime, daynight, innings, "
bulkGameInsertQuery += "tiebreaker, usedh, htbf, timeofgame, attendance, fieldcond, precip, sky, temp, winddir, windspeed, "
bulkGameInsertQuery += "oscorer, forfeit, suspend, umphome, ump1b, ump2b, ump3b, umplf, umprf, wp, lp, save, gametype, "
bulkGameInsertQuery += "vruns, hruns, wteam, lteam, line, batteries, lineups, box, pbp, season, manually_generated) VALUES "

for i in range(len(gameQueries)):
    curGameQuery = gameQueries[i]
    
    bulkGameInsertQuery += curGameQuery

    if i != len(gameQueries) - 1:
        bulkGameInsertQuery += ", "

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(bulkGameInsertQuery)

connection.commit()

cursor    .close()
connection.close()

In [ ]:
bulkBatterInsertQuery  = "INSERT INTO batters (gid, id, team, b_lp, b_seq, stattype, b_pa, b_ab, b_r, b_h, b_d, b_t, b_hr, b_rbi, "
bulkBatterInsertQuery += "b_sh, b_sf, b_hbp, b_w, b_iw, b_k, b_sb, b_cs, b_gdp, b_xi, b_roe, dh, ph, pr, date, number, site, vishome, "
bulkBatterInsertQuery += "opp, win, loss, tie, gametype, box, pbp, manually_generated) VALUES "

for i in range(len(batterQueries)):
    curBatterQuery = batterQueries[i]
    
    bulkBatterInsertQuery += curBatterQuery

    if i != len(batterQueries) - 1:
        bulkBatterInsertQuery += ", "

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(bulkBatterInsertQuery)

connection.commit()

cursor    .close()
connection.close()

In [ ]:
bulkPitcherInsertQuery  = "INSERT INTO pitching (gid, id, team, p_seq, stattype, p_ipouts, p_noout, p_bfp, p_h, p_d, p_t, "
bulkPitcherInsertQuery += "p_hr, p_r, p_er, p_w, p_iw, p_k, p_hbp, p_wp, p_bk, p_sh, p_sf, p_sb, p_cs, p_pb, wp, lp, "
bulkPitcherInsertQuery += "save, p_gs, p_gf, p_cg, date, number, site, vishome, opp, win, loss, tie, gametype, box, pbp, manually_generated) VALUES "

for i in range(len(pitcherQueries)):
    curPitcherQuery = pitcherQueries[i]
    
    bulkPitcherInsertQuery += curPitcherQuery

    if i != len(pitcherQueries) - 1:
        bulkPitcherInsertQuery += ", "

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(bulkPitcherInsertQuery)

connection.commit()

cursor    .close()
connection.close()

In [ ]:
current_datetime = datetime.now()
current_time = current_datetime.strftime("%H:%M:%S")
print("Current time:", current_time)

bulkPlayInsertQuery  = "INSERT INTO plays (gid, event, inning, top_bot, vis_home, ballpark, batteam, pitteam, "
bulkPlayInsertQuery += "batter, pitcher, lp, bat_f, bathand, pithand, count, pitches, nump, pa, ab, single, "
bulkPlayInsertQuery += "double, triple, hr, sh, sf, hbp, walk, iw, k, xi, oth, othout, noout, bip, bunt, ground, fly, line, "
bulkPlayInsertQuery += "gdp, othdp, tp, wp, pb, bk, oa, di, sb2, sb3, sbh, cs2, cs3, csh, pko1, pko2, pko3, k_safe, e1, e2, "
bulkPlayInsertQuery += "e3, e4, e5, e6, e7, e8, e9, outs_pre, outs_post, br1_pre, br2_pre, br3_pre, br1_post, br2_post, "
bulkPlayInsertQuery += "br3_post, run_b, run1, run2, run3, prun1, prun2, prun3, runs, rbi, er, tur, f2, f3, f4, f5, f6, f7, f8, "
bulkPlayInsertQuery += "f9, po0, po1, po2, po3, po4, po5, po6, po7, po8, po9, a1, a2, a3, a4, a5, a6, a7, a8, a9, batout1, batout2, "
bulkPlayInsertQuery += " batout3, brout_b, brout1, brout2, brout3, firstf, loc, hittype, dpopp, pivot, pn, date, gametype, pbp, "
bulkPlayInsertQuery += "manually_generated, atBatIndex) VALUES "

for i in range(len(playQueries)):
    curPlayQuery = playQueries[i]
    
    bulkPlayInsertQuery += curPlayQuery

    if i != len(playQueries) - 1:
        bulkPlayInsertQuery += ", "

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(bulkPlayInsertQuery)

connection.commit()

cursor    .close()
connection.close()

current_datetime = datetime.now()
current_time = current_datetime.strftime("%H:%M:%S")
print("Current time:", current_time)

In [ ]:
current_datetime = datetime.now()
current_time = current_datetime.strftime("%H:%M:%S")
print("Current time:", current_time)

bulkPitchLogInsertQuery  = "INSERT INTO pitch_log (gid, bid, pid, atBatIndex, call, ballColor, trailColor, inPlay, isStrike, isBall, pitchType, "
bulkPitchLogInsertQuery += "isOut, hasReview, balls, strikes, outs, pitchStartSpeed, pitchEndSpeed, strikeZoneTop, strikeZoneBottom, aY, "
bulkPitchLogInsertQuery += "aZ, pfxX, pfxZ, pX, pZ, vX0, vY0, vZ0, x, y, x0, y0, z0, aX, breakAngle, breakLength, breakY, breakVertical, "
bulkPitchLogInsertQuery += "breakVerticalInduced, breakHorizontal, spinRate, spinDirection, pitchNumber, hitLaunchAngle, hitLaunchSpeed, "
bulkPitchLogInsertQuery += "hitTotalDistance, hitTrajectory, hitHardness, hitLocation, hitCoordX, hitCoordY) VALUES "

for i in range(len(pitchLogQueries)):
    curPitchLogQuery = pitchLogQueries[i]
    
    bulkPitchLogInsertQuery += curPitchLogQuery

    if i != len(pitchLogQueries) - 1:
        bulkPitchLogInsertQuery += ", "

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(bulkPitchLogInsertQuery)

connection.commit()

cursor    .close()
connection.close()

current_datetime = datetime.now()
current_time = current_datetime.strftime("%H:%M:%S")
print("Current time:", current_time)

In [ ]:
gameQueries     = []
batterQueries   = []
pitcherQueries  = []
playQueries     = []
pitchLogQueries = []
gameLinks       = []

indexLock = threading.Lock()

dates = gameScheduleJson["dates"]

current_datetime = datetime.now()
current_time = current_datetime.strftime("%H:%M:%S")
print("Current time:", current_time)

for date in dates:
   games = date["games"]

   for game in games:
       link        = game["link"]
       curGameType = game["gameType"]
       status      = game["status"]

       codedGameState = status["codedGameState"]
       
       if curGameType == 'R' and codedGameState == 'F':
           gameLinks.append(link)

with ThreadPoolExecutor(max_workers=10, thread_name_prefix="Worker") as executor:
    for curGameLink in gameLinks:
        executor.submit(getMlbDataFromGame, curGameLink)

current_datetime = datetime.now()
current_time = current_datetime.strftime("%H:%M:%S")
print("Current time:", current_time)

In [ ]:
print(len(pitcherQueries))

In [ ]:
game might have been rescheduled to a date in the future?...

    "status": {
      "abstractGameState": "Final",
      "codedGameState": "F",
      "detailedState": "Final",
      "statusCode": "F",
      "startTimeTBD": false,
      "abstractGameCode": "F"
    },

    "status": {
      "abstractGameState": "Preview",
      "codedGameState": "S",
      "detailedState": "Scheduled",
      "statusCode": "S",
      "startTimeTBD": false,
      "abstractGameCode": "P"
    },
